# Indicator clustering (NOI + PRISM)

K-means on **medium+** indicators that appear in both production NOI forecasts and PRISM scores.

Features: partner count, frequencies (1/7/30d), NOI probabilities (1/7/14/30/45d).
PRISM score and severity are descriptors only (they dominate if used as features).

Companion notebook: `NextObservedRelationships.ipynb` (prediction correlations / associations).

Outputs: `htoc_ml/analysis/_outputs/indicator_clustering/`


## Load NOI forecasts


In [ ]:
import pandas as pd
from pathlib import Path
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
from htoc.core.eval.metrics import parse_probability_percent

def load_one(path: Path) -> pd.DataFrame:
    return pd.read_csv(path).assign(Partner=path.parent.name)

main_folder = Path(r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions")
allowed_folders = ["CDC", "CMS", "DHA", "FDA", "HHS", "HRSA", "IHS", "NIH", "OS", "VA"]
file_date = date.today().strftime("%Y%m%d")
paths = [
    main_folder / p / f"{p}_output_{file_date}.csv"
    for p in allowed_folders
    if (main_folder / p / f"{p}_output_{file_date}.csv").is_file()
]
if not paths:
    raise FileNotFoundError(f"No partner CSVs for {file_date} under {main_folder}")

frames = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(load_one, path): path for path in paths}
    for fut in as_completed(futures):
        frames.append(fut.result())

NOI_df = pd.concat(frames, ignore_index=True)
NOI_df = NOI_df.rename(columns={"ensemble_45d": "Probability: 45-Day"})
PROB_SRC = {
    "noi_prob_1": "Probability: 1-Day",
    "noi_prob_7": "Probability: 7-Day",
    "noi_prob_14": "Probability: 14-Day",
    "noi_prob_30": "Probability: 30-Day",
    "noi_prob_45": "Probability: 45-Day",
}
for col, src in PROB_SRC.items():
    if src in NOI_df.columns:
        NOI_df[col] = parse_probability_percent(NOI_df[src]) / 100.0

NOI_df["Indicator"] = NOI_df["Indicator"].astype("string").str.strip()
NOI_df["Partner"] = NOI_df["Partner"].astype("string").str.strip()
print(f"{len(paths)} partner files for {file_date}: {NOI_df.shape}")
NOI_df.head()


## Load PRISM scores


In [ ]:
prism_excel_path = Path(r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx")
prism_df = pd.read_excel(prism_excel_path, sheet_name="PRISM Scores")
prism_df = prism_df.rename(columns=str.strip)
keep = [c for c in ["Indicator", "Severity", "PRISM Score", "Partners"] if c in prism_df.columns]
prism_df = prism_df[keep].copy()
prism_df["Indicator"] = prism_df["Indicator"].astype(str).str.strip()
prism_df.head()


## Cluster indicators


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

SEV_KEEP = ("medium", "high", "critical")
PROB_COLS = ["noi_prob_1", "noi_prob_7", "noi_prob_14", "noi_prob_30", "noi_prob_45"]
FREQ_COLS = ["freq_1", "freq_7", "freq_30"]

ind = (
    NOI_df.groupby("Indicator", as_index=False)
    .agg(
        n_partners=("Partner", "nunique"),
        freq_1=("Frequency (1d)", "max"),
        freq_7=("Frequency (7d)", "max"),
        freq_30=("Frequency (30d)", "max"),
        **{c: (c, "max") for c in PROB_COLS if c in NOI_df.columns},
    )
)
prism = prism_df.copy()
prism["Severity"] = prism["Severity"].astype(str).str.strip().str.lower()
prism["PRISM Score"] = pd.to_numeric(prism["PRISM Score"], errors="coerce")

merged_df = ind.merge(prism[["Indicator", "Severity", "PRISM Score"]], on="Indicator", how="inner")
merged_df = merged_df[merged_df["Severity"].isin(SEV_KEEP)].copy()

feat_cols = ["n_partners"] + [c for c in FREQ_COLS + PROB_COLS if c in merged_df.columns]
desc_cols = ["PRISM Score", "n_partners"] + [c for c in FREQ_COLS + PROB_COLS if c in merged_df.columns]
X = merged_df[feat_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
X_scaled = StandardScaler().fit_transform(X)
merged_df["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaled)

print(f"{len(merged_df)} medium+ indicators")
print("K-means features:", feat_cols)
display(merged_df["cluster"].value_counts().sort_index().rename("n_indicators"))
display(merged_df.groupby("cluster")[desc_cols].mean().round(3))
display(pd.crosstab(merged_df["cluster"], merged_df["Severity"]))


## Visualize clusters


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

plt.close("all")
xy = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
clusters = sorted(merged_df["cluster"].unique())
prob_plot = [c for c in ["noi_prob_7", "noi_prob_14", "noi_prob_30", "noi_prob_45"] if c in merged_df.columns]
means = merged_df.groupby("cluster")[["n_partners", "freq_1", "freq_7", "freq_30", "PRISM Score"] + prob_plot].mean()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
ax = axes[0, 0]
for c in clusters:
    m = merged_df["cluster"].to_numpy() == c
    ax.scatter(xy[m, 0], xy[m, 1], s=18, alpha=0.55, label=f"Cluster {c} (n={int(m.sum())})")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of scaled features")
ax.legend(frameon=False, fontsize=8)

ax = axes[0, 1]
for c in clusters:
    m = merged_df["cluster"] == c
    ax.scatter(merged_df.loc[m, "n_partners"], merged_df.loc[m, "noi_prob_7"], s=18, alpha=0.55, label=f"Cluster {c}")
ax.set_xlabel("n_partners")
ax.set_ylabel("noi_prob_7")
ax.set_title("Spread vs 7-day probability")
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 0]
for c in clusters:
    m = merged_df["cluster"] == c
    ax.scatter(merged_df.loc[m, "PRISM Score"], merged_df.loc[m, "noi_prob_7"], s=18, alpha=0.55, label=f"Cluster {c}")
ax.set_xlabel("PRISM Score")
ax.set_ylabel("noi_prob_7")
ax.set_title("PRISM vs 7-day probability")
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 1]
x = np.arange(len(means))
w = 0.2
for i, col in enumerate(prob_plot):
    ax.bar(x + (i - 1.5) * w, means[col], w, label=col)
ax.set_xticks(x)
ax.set_xticklabels([f"C{c}" for c in means.index])
ax.set_ylabel("Mean NOI probability")
ax.set_ylim(0, 1)
ax.set_title("Mean NOI probabilities")
ax.legend(frameon=False, fontsize=8)

fig.suptitle("Medium+ indicator clusters", y=1.01)
fig.tight_layout()
plt.show()
plt.close(fig)


## Save


In [ ]:
out = Path(r"h:\HTOC\htoc_ml\analysis\_outputs\indicator_clustering")
out.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(out / "indicator_clusters.csv", index=False)
print("wrote", out / "indicator_clusters.csv")
